In [ ]:
!pip install lightfm scikit-learn pandas numpy scipy


In [ ]:
import pandas as pd
import numpy as np
from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.evaluation import precision_at_k, recall_at_k
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack

In [ ]:
ratings = pd.read_csv('ratings.csv', sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])

movies = pd.read_csv('movies.csv', sep='|', encoding='latin-1', names=[
    'item_id', 'title', 'release_date', 'video_release_date', 'IMDb_URL',
    'unknown', 'Action', 'Adventure', 'Animation', "Children's", 'Comedy',
    'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
    'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'
])


In [ ]:
# Select only genre columns
genre_cols = movies.columns[5:]
movies['genres'] = movies[genre_cols].apply(lambda x: ','.join(genre_cols[x == 1]), axis=1)

# TF-IDF vectorize genres
vectorizer = TfidfVectorizer()
genre_features = vectorizer.fit_transform(movies['genres'])

# Store mapping for LightFM later
movie_id_map = dict(zip(movies['item_id'], movies.index))


In [ ]:
dataset = Dataset()
dataset.fit(
    (x for x in ratings['user_id']),
    (x for x in ratings['item_id'])
)

# Build user-item interactions matrix
(interactions, weights) = dataset.build_interactions(
    [(row['user_id'], row['item_id'], row['rating']) for index, row in ratings.iterrows()]
)
